# DLC Microbead Glaucoma Saccade Workflow

This notebook outlines and implements a DeepLabCut + data science workflow for detecting whether a microbead glaucoma model alters visually evoked saccades during drifting grating stimulation.

## Experimental design

```text
Control group: ~20 mice
Microbead glaucoma-model group: ~20 mice
Microbead timing: ~14 days after injection
Recording: one eye per animal
Stimuli: drifting gratings varying contrast, brightness, velocity, and width/spatial resolution
```

## Primary outcome

```text
saccade_count per mouse per stimulus condition
```

## Main biological question

```text
Do microbead glaucoma-model mice show fewer or altered saccades compared with control mice,
and does the effect depend on contrast, brightness, velocity, or grating width?
```

# 1. Brief explanation of DeepLabCut

DeepLabCut is a supervised deep-learning pose estimation tool.

You manually label keypoints on selected frames, such as:

```text
pupil_center
pupil_left_edge
pupil_right_edge
pupil_top_edge
pupil_bottom_edge
eyelid landmarks
```

DLC trains a convolutional neural network to learn:

```text
video frame → bodypart x/y coordinates + likelihood score
```

## Neural network architecture

DLC usually uses a convolutional neural network backbone such as ResNet or similar architectures. The network extracts spatial image features and predicts heatmaps for each bodypart. A heatmap is a probability-like map showing where a keypoint is likely to be. DLC converts the heatmap peak into x/y coordinates and reports a likelihood/confidence value.

Conceptually:

```text
eye video frame
    ↓
CNN feature extractor
    ↓
bodypart heatmaps
    ↓
x/y coordinates
    ↓
likelihood score
```

## Training

Training compares predicted keypoint positions to manually labeled positions and adjusts the neural network weights to reduce localization error.

## Prediction

After training, DLC analyzes every frame of new videos and outputs `.h5` and/or `.csv` files containing x, y, and likelihood values for each labeled keypoint.

# 2. Libraries needed and purpose

| Library | Purpose |
|---|---|
| `pathlib` | Handle file/folder paths |
| `re` | Match filenames/recording IDs |
| `numpy` | Numerical calculations, displacement, velocity |
| `pandas` | Load and save CSV/HDF tables |
| `matplotlib` | Plot eye traces and summary figures |
| `seaborn` | Group/stimulus statistical plots |
| `scipy.signal` | Detect saccades using peak detection |
| `deeplabcut` | Train DLC model and analyze videos |
| `statsmodels` | Poisson/negative-binomial regression for saccade counts |
| `cv2` | Optional video metadata access |

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import find_peaks

import statsmodels.formula.api as smf
import statsmodels.api as sm

try:
    import deeplabcut
    DLC_AVAILABLE = True
except ImportError:
    DLC_AVAILABLE = False

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

sns.set_context("notebook")
sns.set_style("whitegrid")

print("DeepLabCut available:", DLC_AVAILABLE)
print("OpenCV available:", CV2_AVAILABLE)

# 3. Inputs and expected folder structure

Expected project folder:

```text
DLC_PROJECT/
    config.yaml
    videos/
        *.mp4
        DLC prediction *.h5
    stimulus/
        TSeries_*/
            ttl_combined_summary_*.csv
    metadata/
        animal_metadata.csv
        stimulus_metadata.csv
    aligned_analysis/
        exports/
        saccades/
        summaries/
        figures/
```

## `animal_metadata.csv`

| column | description |
|---|---|
| `mouse_id` | animal identifier |
| `group` | `control` or `microbead` |
| `video_file` | original video filename |
| `eye_side` | recorded eye |
| `days_after_injection` | expected ~14 for microbead group |

## `stimulus_metadata.csv`

| column | description |
|---|---|
| `stimulus_index` | stimulus number |
| `contrast` | grating contrast |
| `brightness` | brightness level |
| `velocity` | drift speed |
| `width` | grating width/spatial resolution |
| `stimulus_name` | optional label |

In [ ]:
project_path = Path("../DLC_PROJECT")

config_path = project_path / "config.yaml"
video_dir = project_path / "videos"
stimulus_dir = project_path / "stimulus"
metadata_dir = project_path / "metadata"

output_dir = project_path / "aligned_analysis"
export_dir = output_dir / "exports"
saccade_dir = output_dir / "saccades"
summary_dir = output_dir / "summaries"
fig_dir = output_dir / "figures"

for folder in [output_dir, export_dir, saccade_dir, summary_dir, fig_dir]:
    folder.mkdir(parents=True, exist_ok=True)

animal_metadata_path = metadata_dir / "animal_metadata.csv"
stimulus_metadata_path = metadata_dir / "stimulus_metadata.csv"

print("Project path:", project_path)
print("Config path:", config_path)

# 4. DLC training and prediction commands

These are intentionally commented out so the notebook does not start training accidentally.

## What this step accomplishes

1. Creates a training dataset from manually labeled frames.
2. Trains the DLC neural network.
3. Evaluates network performance.
4. Runs prediction on all videos.
5. Creates labeled videos for quality control.

In [ ]:
if DLC_AVAILABLE and config_path.exists():

    video_list = sorted(video_dir.glob("*.mp4"))
    video_list_str = [str(v) for v in video_list]

    print("Videos found:", len(video_list))

    # Create training dataset after labeling frames.
    # deeplabcut.create_training_dataset(str(config_path))

    # Train network.
    # deeplabcut.train_network(
    #     str(config_path),
    #     shuffle=1,
    #     displayiters=100,
    #     saveiters=1000,
    #     maxiters=20000
    # )

    # Evaluate trained network.
    # deeplabcut.evaluate_network(
    #     str(config_path),
    #     plotting=True,
    #     show_errors=True
    # )

    # Analyze all videos.
    # deeplabcut.analyze_videos(
    #     str(config_path),
    #     videos=video_list_str,
    #     shuffle=1,
    #     save_as_csv=True
    # )

    # Create labeled QC videos.
    # deeplabcut.create_labeled_video(
    #     str(config_path),
    #     videos=video_list_str,
    #     shuffle=1,
    #     draw_skeleton=True,
    #     pcutoff=0.3
    # )

else:
    print("DLC unavailable or config.yaml not found. This is expected if running analysis only.")

# 5. Load DLC eye metrics

This function reads a DLC `.h5` prediction file and computes:

```text
pupil_x
pupil_y
movement/displacement
horizontal pupil diameter
vertical pupil diameter
average pupil diameter
tracking quality
```

Output: one row per video frame.

In [ ]:
def load_dlc_eye_metrics(h5_path):
    h5_path = Path(h5_path)

    df = pd.read_hdf(h5_path)

    if not isinstance(df.columns, pd.MultiIndex):
        raise ValueError(f"{h5_path.name} does not have DLC MultiIndex columns.")

    scorer = df.columns.get_level_values(0)[0]

    def get(bp, coord):
        return df[(scorer, bp, coord)]

    required_bodyparts = [
        "pupil_center",
        "pupil_left_edge",
        "pupil_right_edge",
        "pupil_top_edge",
        "pupil_bottom_edge"
    ]

    for bp in required_bodyparts:
        if (scorer, bp, "x") not in df.columns:
            raise ValueError(f"Missing bodypart: {bp}")

    pupil_x = get("pupil_center", "x")
    pupil_y = get("pupil_center", "y")

    movement = np.sqrt(
        (pupil_x - pupil_x.iloc[0]) ** 2
        + (pupil_y - pupil_y.iloc[0]) ** 2
    )

    diameter_h = np.abs(
        get("pupil_right_edge", "x")
        - get("pupil_left_edge", "x")
    )

    diameter_v = np.abs(
        get("pupil_bottom_edge", "y")
        - get("pupil_top_edge", "y")
    )

    diameter_avg = (diameter_h + diameter_v) / 2

    likelihoods = pd.concat(
        [get(bp, "likelihood") for bp in required_bodyparts],
        axis=1
    )

    tracking_quality = likelihoods.mean(axis=1)

    out = pd.DataFrame({
        "frame_number": np.arange(len(df)),
        "pupil_x": pupil_x.values,
        "pupil_y": pupil_y.values,
        "movement": movement.values,
        "diameter_h": diameter_h.values,
        "diameter_v": diameter_v.values,
        "diameter_avg": diameter_avg.values,
        "tracking_quality": tracking_quality.values,
        "video_name": h5_path.stem
    })

    return out

# 6. Align eye traces to drifting grating stimuli

Input:

```text
DLC frame-level eye metrics
TTL/stimulus frame CSV
stimulus metadata CSV
```

Output:

```text
frame_number
pupil_x
pupil_y
movement
diameter_avg
stimulus_index
contrast
brightness
velocity
width
```

In [ ]:
def align_eye_metrics_to_stimulus(
    eye_df,
    stimulus_frame_df,
    stimulus_metadata_df=None
):
    aligned = eye_df.copy()
    aligned["stimulus_index"] = np.nan

    for _, row in stimulus_frame_df.iterrows():

        if str(row["stimulus_index"]).upper() == "CAMERA":
            continue

        stim_idx = int(row["stimulus_index"])
        start = int(row["stimulus_on_frame"])
        stop = int(row["stimulus_off_frame"])

        mask = (
            (aligned["frame_number"] >= start)
            & (aligned["frame_number"] < stop)
        )

        aligned.loc[mask, "stimulus_index"] = stim_idx

    if stimulus_metadata_df is not None:
        aligned = aligned.merge(
            stimulus_metadata_df,
            on="stimulus_index",
            how="left"
        )

    return aligned

# 7. Detect saccades

Saccades are detected as peaks in movement/displacement.

Tune these parameters after visual inspection:

```text
saccade_min_amp
saccade_min_distance
saccade_min_prominence
min_tracking_quality
```

In [ ]:
def detect_saccades(
    aligned_df,
    saccade_min_amp=5.0,
    saccade_min_distance=5,
    saccade_min_prominence=2.0,
    min_tracking_quality=0.6
):
    df = aligned_df.copy()

    df["saccade"] = False
    df["saccade_amplitude_px"] = np.nan

    movement = df["movement"].copy()
    movement[df["tracking_quality"] < min_tracking_quality] = np.nan

    movement_interp = movement.interpolate(
        limit_direction="both"
    )

    peaks, properties = find_peaks(
        movement_interp.values,
        height=saccade_min_amp,
        distance=saccade_min_distance,
        prominence=saccade_min_prominence
    )

    df.loc[peaks, "saccade"] = True
    df.loc[peaks, "saccade_amplitude_px"] = movement_interp.iloc[peaks].values

    return df

# 8. Match DLC files to stimulus files

Customize this if your naming scheme differs.

Best practice: include explicit paths in `animal_metadata.csv`.

This default matcher searches for shared numeric tokens between video/DLC filename and TTL/stimulus CSV.

In [ ]:
def find_first_matching_stimulus_csv(stimulus_dir, video_name):
    video_stem = Path(video_name).stem
    candidate_csvs = list(Path(stimulus_dir).glob("**/ttl_combined_summary_*.csv"))

    tokens = re.findall(r"\d{4,}", video_stem)

    for token in tokens:
        for csv_path in candidate_csvs:
            if token in csv_path.name or token in str(csv_path.parent):
                return csv_path

    return None


def process_single_dlc_file(
    h5_path,
    stimulus_metadata_df=None
):
    eye_df = load_dlc_eye_metrics(h5_path)

    stim_csv = find_first_matching_stimulus_csv(
        stimulus_dir,
        h5_path.name
    )

    if stim_csv is None:
        raise FileNotFoundError(f"No stimulus CSV matched {h5_path.name}")

    stim_frame_df = pd.read_csv(stim_csv)

    aligned = align_eye_metrics_to_stimulus(
        eye_df,
        stim_frame_df,
        stimulus_metadata_df=stimulus_metadata_df
    )

    detected = detect_saccades(aligned)

    return detected

# 9. Load metadata

This step loads animal and stimulus metadata if available.

In [ ]:
if stimulus_metadata_path.exists():
    stimulus_metadata = pd.read_csv(stimulus_metadata_path)
    print("Loaded stimulus metadata:", stimulus_metadata.shape)
else:
    stimulus_metadata = None
    print("No stimulus metadata found.")

if animal_metadata_path.exists():
    animal_metadata = pd.read_csv(animal_metadata_path)
    print("Loaded animal metadata:", animal_metadata.shape)
else:
    animal_metadata = None
    print("No animal metadata found.")

# 10. Batch process prediction files

This cell processes all DLC `.h5` prediction files into aligned frame-level CSVs with saccade labels.

Output:

```text
aligned_analysis/exports/*_aligned_saccades.csv
```

In [ ]:
# Uncomment when your project paths and filename matching are confirmed.

# h5_files = sorted(video_dir.glob("*.h5"))

# for h5_path in h5_files:
#     print("Processing:", h5_path.name)

#     try:
#         detected_df = process_single_dlc_file(
#             h5_path,
#             stimulus_metadata_df=stimulus_metadata
#         )

#         out_csv = export_dir / f"{h5_path.stem}_aligned_saccades.csv"

#         detected_df.to_csv(out_csv, index=False)

#         print("Saved:", out_csv)

#     except Exception as e:
#         print("Skipped:", h5_path.name, "Reason:", e)

# 11. Aggregate by stimulus condition

This converts frame-level data into one row per video/stimulus condition.

This is the main table for statistics.

In [ ]:
def summarize_saccades_by_stimulus(aligned_df):

    rows = []

    stim_data = aligned_df.dropna(subset=["stimulus_index"])

    for stim_idx, stim_df in stim_data.groupby("stimulus_index"):

        stim_df = stim_df.sort_values("frame_number")

        saccade_rows = stim_df[stim_df["saccade"]]

        n_frames = len(stim_df)
        saccade_count = int(stim_df["saccade"].sum())

        if len(saccade_rows) > 0:
            latency = (
                saccade_rows["frame_number"].iloc[0]
                - stim_df["frame_number"].iloc[0]
            )
            mean_amp = saccade_rows["saccade_amplitude_px"].mean()
        else:
            latency = np.nan
            mean_amp = np.nan

        row = {
            "video_name": stim_df["video_name"].iloc[0],
            "stimulus_index": int(stim_idx),
            "n_frames": n_frames,
            "saccade_count": saccade_count,
            "saccades_per_frame": saccade_count / n_frames if n_frames > 0 else np.nan,
            "mean_saccade_amplitude_px": mean_amp,
            "latency_to_first_saccade_frames": latency,
            "mean_pupil_diameter": stim_df["diameter_avg"].mean(),
            "mean_tracking_quality": stim_df["tracking_quality"].mean()
        }

        for col in ["contrast", "brightness", "velocity", "width", "stimulus_name"]:
            if col in stim_df.columns:
                row[col] = stim_df[col].dropna().iloc[0] if stim_df[col].notna().any() else np.nan

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
aligned_csvs = sorted(export_dir.glob("*_aligned_saccades.csv"))
summary_tables = []

for csv_path in aligned_csvs:
    aligned_df = pd.read_csv(csv_path)
    summary = summarize_saccades_by_stimulus(aligned_df)
    summary_tables.append(summary)

if summary_tables:
    stimulus_summary = pd.concat(summary_tables, ignore_index=True)

    if animal_metadata is not None and "video_file" in animal_metadata.columns:
        animal_metadata = animal_metadata.copy()
        animal_metadata["video_stem"] = animal_metadata["video_file"].apply(lambda x: Path(str(x)).stem)
        stimulus_summary["video_stem"] = stimulus_summary["video_name"].apply(lambda x: Path(str(x)).stem)

        stimulus_summary = stimulus_summary.merge(
            animal_metadata,
            on="video_stem",
            how="left"
        )

    summary_path = summary_dir / "saccade_stimulus_summary.csv"
    stimulus_summary.to_csv(summary_path, index=False)

    display(stimulus_summary.head())
    print("Saved:", summary_path)

else:
    print("No aligned saccade CSVs found yet.")

# 12. Plot group and stimulus effects

These figures show whether saccade counts differ by group and stimulus parameters.

In [ ]:
if "stimulus_summary" in globals():

    if "group" in stimulus_summary.columns:
        plt.figure(figsize=(7, 5))

        sns.boxplot(
            data=stimulus_summary,
            x="group",
            y="saccade_count"
        )

        sns.stripplot(
            data=stimulus_summary,
            x="group",
            y="saccade_count",
            color="black",
            alpha=0.35
        )

        plt.title("Saccade count by group")
        plt.tight_layout()

        out_path = fig_dir / "saccade_count_by_group.tiff"
        plt.savefig(out_path, dpi=300, format="tiff", bbox_inches="tight")
        plt.show()

    for stim_var in ["contrast", "brightness", "velocity", "width"]:
        if stim_var in stimulus_summary.columns and "group" in stimulus_summary.columns:
            plt.figure(figsize=(8, 5))

            sns.pointplot(
                data=stimulus_summary,
                x=stim_var,
                y="saccade_count",
                hue="group",
                errorbar="se"
            )

            plt.title(f"Saccade count by {stim_var} and group")
            plt.tight_layout()

            out_path = fig_dir / f"saccade_count_by_{stim_var}_group.tiff"
            plt.savefig(out_path, dpi=300, format="tiff", bbox_inches="tight")
            plt.show()

else:
    print("stimulus_summary has not been created yet.")

# 13. Statistical model for saccade counts

Saccade count is count data, so use Poisson regression as a first model.

Model:

```text
saccade_count ~ group + contrast + brightness + velocity + width
```

For final analysis, a Bayesian hierarchical Poisson or negative-binomial model is preferable because stimulus conditions are repeated within each mouse.

## Output interpretation

| Term | Meaning |
|---|---|
| `group[T.microbead]` | effect of glaucoma model on saccade count |
| `contrast` | effect of grating contrast |
| `velocity` | effect of drift speed |
| `width` | effect of grating spatial width/resolution |
| interaction terms | whether microbead effect depends on stimulus |

In [ ]:
if "stimulus_summary" in globals() and "group" in stimulus_summary.columns:

    model_df = stimulus_summary.dropna(
        subset=["saccade_count", "group"]
    ).copy()

    predictors = ["C(group)"]

    for col in ["contrast", "brightness", "velocity", "width"]:
        if col in model_df.columns and model_df[col].notna().any():
            predictors.append(col)

    formula = "saccade_count ~ " + " + ".join(predictors)

    print("Formula:", formula)

    poisson_model = smf.glm(
        formula=formula,
        data=model_df,
        family=sm.families.Poisson()
    ).fit()

    print(poisson_model.summary())

    model_summary_path = summary_dir / "poisson_saccade_count_model_summary.txt"

    with open(model_summary_path, "w") as f:
        f.write(poisson_model.summary().as_text())

    print("Saved:", model_summary_path)

else:
    print("stimulus_summary with group column not available yet.")

# 14. Recommended final publication-level model

For final analysis:

```text
saccade_count ~ group * contrast + group * brightness + group * velocity + group * width + animal_effect
```

This accounts for:

```text
multiple stimulus conditions within the same mouse
```

Best model types:

```text
Bayesian hierarchical Poisson model
Bayesian hierarchical negative-binomial model
mixed-effects count model
```

The most useful final outputs:

```text
probability microbead mice have fewer saccades
effect size of microbead group
group × contrast interaction
group × velocity interaction
group × width interaction
credible intervals or confidence intervals
```

# 15. Final deliverables

Expected outputs:

```text
trained DLC model
labeled QC videos
confidence overlay videos
DLC prediction h5/csv files
aligned frame-level eye/stimulus CSVs
saccade event CSVs
per-animal/per-stimulus summary CSV
group/stimulus plots
saccade count model summary
```

Main biological interpretation:

```text
If microbead mice show fewer saccades, especially at low contrast,
high velocity, or finer grating width, this suggests glaucoma-model injury
reduces visually evoked oculomotor responses.
```